In [ ]:
import pandas as pd
from modules.entity import Entity

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.neighbors import KDTree
from pyqubo import Array, Constraint, Placeholder
from neal import SimulatedAnnealingSampler
import plotly.graph_objects as go

## データ入力

In [ ]:
ent = Entity('input/sample1.pdb')
ent.generate_monomers()

In [ ]:
monomers = ent.monomers

In [ ]:
ent.to_dataframe()

## 定式化・最適化

In [ ]:
from modules.model_point import OptimizeClusterPoint

### 最適化1: 全ての点をクラスタリング。グループごとに代表点を選択する

In [ ]:
# -------------------------------
# 問題規模
# -------------------------------
n_points = len(monomers)

# -------------------------------
# クラスタリング
# -------------------------------
np.random.seed(0)
n_cluster = 2 # クラスタ数

# 各点の座標
positions = [[m.coordinate.x, m.coordinate.y, m.coordinate.z] for m in monomers]

# Kmeansで、各点の座標情報に基づいてクラスタ番号を採番
kmeans = KMeans(n_clusters=n_cluster, random_state=0)
cluster_labels = kmeans.fit_predict(positions)

In [ ]:
selected_points = []
CLUSTER_MAX_SIZE = 60

for cluster_id in range(n_cluster):
    cluster_indices = np.where(cluster_labels == cluster_id)[0]
    if len(cluster_indices) > CLUSTER_MAX_SIZE:
        cluster_indices = np.random.choice(cluster_indices, CLUSTER_MAX_SIZE, replace=False)
    selected = OptimizeClusterPoint(cluster_indices=cluster_indices, monomers=monomers, positions=positions).optimize_cluster()
    selected_points.extend(selected)
print(f"クラスタ最適化後の選択点数: {len(selected_points)}")
print(f"採用された点: ", selected_points)

In [ ]:
from modules.model_point import OptimizeOrder

In [ ]:
positions_selected = [positions[selected_point] for selected_point in selected_points]
scores_selected = [monomers[selected_point].bfactor for selected_point in selected_points]
orders = OptimizeOrder(positions_selected, scores_selected).optimize_order()
print(f"最終選択された点・順番: {orders}")
print(f"\n最終選択された点の数: {len(orders)}")

In [ ]:
positions_ordered = np.array([positions_selected[order] for order in range(len(orders))])

## 可視化

In [ ]:
fig = go.Figure()

non_selected = list(set(range(n_points)) - set(selected_points))
positions = np.array(positions)
fig.add_trace(go.Scatter3d(
    x=positions[non_selected, 0],
    y=positions[non_selected, 1],
    z=positions[non_selected, 2],
    mode='markers',
    marker=dict(size=1, color='gray', opacity=0.5),
    name='Non-selected'
))

fig.add_trace(go.Scatter3d(
    x=positions_ordered[:, 0],
    y=positions_ordered[:, 1],
    z=positions_ordered[:, 2],
    mode='markers',
    marker=dict(size=2, color='red', opacity=1.0),
    line=dict(color='red', width=2),
    name='Selected (Ordered)'
))

fig.update_layout(title='3D Visualization of Selected Points (Ordered)',
                  scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'))
fig.show()